# 02 — Exploratory Data Analysis (EDA)
**Goal:** Understand the dataset through visualisations. Find key patterns driving churn.

**Key Questions:**
1. What is the overall churn rate?
2. Which contract type has highest churn?
3. Do churners pay more per month?
4. How does tenure correlate with churn?
5. Which internet service type churns most?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

sns.set_theme(style='darkgrid')
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('../data/processed/cleaned_data.csv')
print(f'Shape: {df.shape} | Churn rate: {df["Churn"].mean():.1%}')
df.head(3)

In [ ]:
# ── 1. Churn Distribution ─────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

counts = df['Churn'].value_counts()
ax1.pie(counts, labels=['Retained', 'Churned'], colors=['#6366f1', '#f43f5e'],
        autopct='%1.1f%%', startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
ax1.set_title('Overall Churn Rate', fontsize=13, fontweight='bold')

sns.countplot(data=df, x='Churn', ax=ax2, palette=['#6366f1', '#f43f5e'])
ax2.set_xticklabels(['Retained (0)', 'Churned (1)'])
ax2.set_title('Churn Count', fontsize=13, fontweight='bold')
for p in ax2.patches:
    ax2.annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width()/2., p.get_height()),
                 ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()
print(f"Churn Rate: {df['Churn'].mean():.2%} — Imbalanced dataset!")

In [ ]:
# ── 2. Churn by Contract Type ─────────────────────────
contract_churn = df.groupby('Contract')['Churn'].mean().reset_index()
contract_churn.columns = ['Contract', 'Churn Rate']
contract_churn['Churn Rate %'] = (contract_churn['Churn Rate'] * 100).round(1)

plt.figure(figsize=(8, 4))
bars = plt.bar(contract_churn['Contract'], contract_churn['Churn Rate %'],
               color=['#f43f5e', '#f59e0b', '#22c55e'], edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, contract_churn['Churn Rate %']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val}%', ha='center', fontweight='bold', fontsize=11)
plt.title('Churn Rate by Contract Type', fontsize=13, fontweight='bold')
plt.ylabel('Churn Rate (%)')
plt.tight_layout()
plt.show()

print('\n💡 Key Insight: Month-to-month customers churn ~3x more than two-year contract holders!')

In [ ]:
# ── 3. Tenure vs Churn ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# KDE plot
for val, label, color in [(0, 'Retained', '#6366f1'), (1, 'Churned', '#f43f5e')]:
    axes[0].hist(df[df['Churn']==val]['tenure'], bins=30, alpha=0.6,
                 color=color, label=label, edgecolor='white', linewidth=0.5)
axes[0].set_title('Tenure Distribution by Churn Status', fontweight='bold')
axes[0].set_xlabel('Tenure (months)')
axes[0].legend()

# Box plot
sns.boxplot(data=df, x='Churn', y='tenure', ax=axes[1],
            palette={0: '#6366f1', 1: '#f43f5e'})
axes[1].set_xticklabels(['Retained', 'Churned'])
axes[1].set_title('Tenure Box Plot by Churn Status', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"Churned customers avg tenure: {df[df['Churn']==1]['tenure'].mean():.1f} months")
print(f"Retained customers avg tenure: {df[df['Churn']==0]['tenure'].mean():.1f} months")
print('\n💡 Insight: Churners leave EARLY — median tenure < 10 months vs 38 for retained!')

In [ ]:
# ── 4. Monthly Charges vs Churn ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.violinplot(data=df, x='Churn', y='MonthlyCharges', ax=axes[0],
               palette={0: '#6366f1', 1: '#f43f5e'})
axes[0].set_xticklabels(['Retained', 'Churned'])
axes[0].set_title('Monthly Charges by Churn Status', fontweight='bold')

# Churn rate by internet service
internet_churn = df.groupby('InternetService')['Churn'].mean() * 100
internet_churn.plot(kind='bar', ax=axes[1], color=['#22c55e', '#f43f5e', '#6366f1'], edgecolor='white')
axes[1].set_title('Churn Rate by Internet Service', fontweight='bold')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_xticklabels(internet_churn.index, rotation=0)
for p in axes[1].patches:
    axes[1].annotate(f'{p.get_height():.1f}%', (p.get_x() + p.get_width()/2, p.get_height() + 0.3),
                     ha='center', fontweight='bold')

plt.tight_layout()
plt.show()
print('\n💡 Insight: Fiber optic users pay more AND churn more — price sensitivity!')

In [ ]:
# ── 5. Correlation Heatmap ────────────────────────────
# Encode categoricals for correlation
df_enc = pd.get_dummies(df.drop('customerID', axis=1))
corr = df_enc.corr()['Churn'].drop('Churn').sort_values()

plt.figure(figsize=(10, 8))
colors = ['#f43f5e' if v > 0 else '#6366f1' for v in corr.values]
corr.plot(kind='barh', color=colors, edgecolor='white', linewidth=0.5)
plt.title('Feature Correlation with Churn', fontsize=13, fontweight='bold')
plt.xlabel('Pearson Correlation')
plt.axvline(0, color='white', linewidth=0.5)
plt.tight_layout()
plt.show()

print('Top POSITIVE correlations (increase churn):')
print(corr.tail(5))
print('\nTop NEGATIVE correlations (decrease churn):')
print(corr.head(5))

In [ ]:
# ── 6. Churn by Senior Citizen ────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, ['SeniorCitizen', 'Partner', 'Dependents']):
    churn_rate = df.groupby(col)['Churn'].mean() * 100
    churn_rate.plot(kind='bar', ax=ax, color=['#6366f1', '#f43f5e'], edgecolor='white', rot=0)
    ax.set_title(f'Churn Rate by {col}', fontweight='bold')
    ax.set_ylabel('Churn Rate (%)')
    for p in ax.patches:
        ax.annotate(f'{p.get_height():.1f}%',
                    (p.get_x() + p.get_width()/2, p.get_height() + 0.3), ha='center')

plt.tight_layout()
plt.show()
print('\n💡 Insight: Senior citizens churn more. Customers with partners/dependents churn less (more stable).')

In [ ]:
# ── 7. Summary: Key EDA Findings ──────────────────────
print('='*60)
print('KEY EDA FINDINGS')
print('='*60)
findings = [
    f"Overall churn rate: {df['Churn'].mean():.1%} (imbalanced)",
    f"Month-to-month churn: {df[df['Contract']=='Month-to-month']['Churn'].mean():.1%}",
    f"Two-year contract churn: {df[df['Contract']=='Two year']['Churn'].mean():.1%}",
    f"Churners avg tenure: {df[df['Churn']==1]['tenure'].mean():.1f} months",
    f"Retained avg tenure: {df[df['Churn']==0]['tenure'].mean():.1f} months",
    f"Fiber optic churn rate: {df[df['InternetService']=='Fiber optic']['Churn'].mean():.1%}",
    f"Electronic check churn: {df[df['PaymentMethod']=='Electronic check']['Churn'].mean():.1%}",
]
for f in findings:
    print(f'  ✅ {f}')